# S10 - Transformación de fechas, texto y discretización

**Estudiante:** Diego Sandoval  
**Asignatura:** Preprocesamiento de Datos

En este notebook se aplican transformaciones al dataset codificado de la Semana 9:

- Transformación de variables temporales.
- Transformación básica de texto.
- Discretización de `edad_cliente`.
- Transformación logarítmica de `monto_compra`.
- Transformación Box-Cox, cuando se cumplen sus requisitos.
- Comparación del sesgo antes y después.
- Construcción y exportación del dataset final.

> **Nota:** Las transformaciones que deben existir antes de dividir el dataset se realizan antes de crear `df_entrenamiento` y `df_prueba`.

## 1. Carga del dataset

Se carga el archivo generado en la Semana 9. Las transformaciones de esta semana se agregan como nuevas columnas y se conservan las variables originales.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("S09_PD_Sandoval_DatasetCodificado.csv")

print("Dimensiones iniciales:", df.shape)

Dimensiones iniciales: (518001, 53)


In [2]:
df[[
    "fecha_pedido",
    "comentario_cliente",
    "edad_cliente",
    "monto_compra"
]].head()

,fecha_pedido,comentario_cliente,edad_cliente,monto_compra
0,2025-07-10,"Todo perfecto, llegó a tiempo",62,366000.0
1,NaN,El producto llegó dañado,42,302100.0
2,2025-11-17,No era lo que esperaba,70,626400.0
3,2025-08-02,Rápido y sin problemas,69,0.0
4,NaN,El producto llegó dañado,22,772000.0


## 2. Transformación de fechas

`fecha_pedido` se convierte explícitamente a tipo fecha para poder utilizar el accesor `.dt` y extraer información temporal.

In [3]:
df["fecha_pedido"] = pd.to_datetime(df["fecha_pedido"])

print("Tipo de fecha_pedido:", df["fecha_pedido"].dtype)

Tipo de fecha_pedido: datetime64[ns]


## 3. Variables temporales extraídas

A partir de `fecha_pedido` se crean las cuatro variables solicitadas:

- `mes_pedido`
- `trimestre_pedido`
- `dia_semana_pedido`
- `es_fin_semana_pedido`

In [4]:
df["mes_pedido"] = df["fecha_pedido"].dt.month
df["trimestre_pedido"] = df["fecha_pedido"].dt.quarter
df["dia_semana_pedido"] = df["fecha_pedido"].dt.day_name()
df["es_fin_semana_pedido"] = df["fecha_pedido"].dt.dayofweek >= 5

In [5]:
df[[
    "fecha_pedido",
    "mes_pedido",
    "trimestre_pedido",
    "dia_semana_pedido",
    "es_fin_semana_pedido"
]].head(10)

,fecha_pedido,mes_pedido,trimestre_pedido,dia_semana_pedido,es_fin_semana_pedido
0,2025-07-10,7.0,3.0,Thursday,False
1,NaT,NaN,NaN,NaN,False
2,2025-11-17,11.0,4.0,Monday,False
3,2025-08-02,8.0,3.0,Saturday,True
4,NaT,NaN,NaN,NaN,False
5,NaT,NaN,NaN,NaN,False
6,2025-09-10,9.0,3.0,Wednesday,False
7,2025-11-26,11.0,4.0,Wednesday,False
8,2025-04-06,4.0,2.0,Sunday,True
9,NaT,NaN,NaN,NaN,False


### Variables temporales extraídas

Se crearon **4 columnas nuevas** a partir de `fecha_pedido`:

- `mes_pedido`: mes en el que se realizó el pedido.
- `trimestre_pedido`: trimestre del año.
- `dia_semana_pedido`: día de la semana.
- `es_fin_semana_pedido`: indica con `True` o `False` si el pedido ocurrió sábado o domingo.

## 4. Transformación básica de texto

Primero se reemplazan los valores faltantes de `comentario_cliente` por una cadena vacía. Esto permite realizar operaciones de texto sin mantener `NaN` en las variables derivadas.

In [6]:
print("Valores faltantes antes:", df["comentario_cliente"].isna().sum())

df["comentario_cliente"] = df["comentario_cliente"].fillna("")

print("Valores faltantes después:", df["comentario_cliente"].isna().sum())

Valores faltantes antes: 73559
Valores faltantes después: 0


## 5. Variables de texto extraídas

Se crean cuatro variables:

- `longitud_comentario`: cantidad de caracteres.
- `num_palabras_comentario`: cantidad de palabras.
- `tiene_comentario`: indica si existe un comentario.
- `menciona_demora`: detecta términos asociados con demoras o retrasos.

In [7]:
df["longitud_comentario"] = df["comentario_cliente"].str.len()

df["num_palabras_comentario"] = (
    df["comentario_cliente"].str.split().str.len()
)

df["tiene_comentario"] = df["comentario_cliente"].str.len() > 0

df["menciona_demora"] = df["comentario_cliente"].str.contains(
    "tarde|demora|retraso|no llegó",
    case=False,
    na=False
)

In [8]:
df[[
    "comentario_cliente",
    "longitud_comentario",
    "num_palabras_comentario",
    "tiene_comentario",
    "menciona_demora"
]].head(10)

,comentario_cliente,longitud_comentario,num_palabras_comentario,tiene_comentario,menciona_demora
0,"Todo perfecto, llegó a tiempo",29,5,True,False
1,El producto llegó dañado,24,4,True,False
2,No era lo que esperaba,22,5,True,False
3,Rápido y sin problemas,22,4,True,False
4,El producto llegó dañado,24,4,True,False
5,El domiciliario fue muy amable,30,5,True,False
6,"Excelente calidad, volvería a comprar",37,5,True,False
7,"Excelente calidad, volvería a comprar",37,5,True,False
8,Llegó incompleto el pedido,26,4,True,False
9,,0,0,False,False


In [9]:
print("Distribución de menciona_demora:")
print(df["menciona_demora"].value_counts())

Distribución de menciona_demora:
menciona_demora
False    518001
Name: count, dtype: int64


### Variables de texto extraídas

- `longitud_comentario` mide el número de caracteres.
- `num_palabras_comentario` cuenta las palabras.
- `tiene_comentario` identifica si el cliente escribió algo.
- `menciona_demora` identifica si aparecen términos relacionados con retrasos o demoras.

Estas son características básicas del texto; no representan un análisis de sentimiento.

## 6. Discretización de `edad_cliente`

Se transforma la edad exacta en grupos definidos por reglas fijas de negocio:

- 18 a 25
- 26 a 35
- 36 a 45
- 46 a 60
- 61 y más

Los límites son reglas fijas (`17, 25, 35, 45, 60, 100`), por lo que no se calculan a partir de los datos.

In [10]:
df["grupo_edad"] = pd.cut(
    df["edad_cliente"],
    bins=[17, 25, 35, 45, 60, 100],
    labels=[
        "18 a 25",
        "26 a 35",
        "36 a 45",
        "46 a 60",
        "61 y más"
    ]
)

In [11]:
print("Distribución de grupo_edad:")
print(df["grupo_edad"].value_counts(dropna=False).sort_index())

Distribución de grupo_edad:
grupo_edad
18 a 25      67226
26 a 35      84190
36 a 45      83635
46 a 60     126723
61 y más    126164
NaN          30063
Name: count, dtype: int64


In [12]:
edades_fuera_rango = (
    (df["edad_cliente"] < 18) |
    (df["edad_cliente"] > 100)
).sum()

print("Registros con edad fuera del rango 18-100:", edades_fuera_rango)

if edades_fuera_rango > 0:
    print("Observación: esos registros quedan sin grupo porque están fuera de los bins definidos.")
else:
    print("Todas las edades están dentro del rango definido.")

Registros con edad fuera del rango 18-100: 30063
Observación: esos registros quedan sin grupo porque están fuera de los bins definidos.


## 7. Sesgo y transformación logarítmica de `monto_compra`

Primero se mide el sesgo de la variable original. Después se aplica `np.log1p()` sobre el dataset completo.

La transformación logarítmica es una fórmula fija: no necesita aprender parámetros mediante `fit`.

In [13]:
sesgo_original = df["monto_compra"].skew()

print("Sesgo original de monto_compra:", sesgo_original)
print("Valores cero:", (df["monto_compra"] == 0).sum())
print("Valores negativos:", (df["monto_compra"] < 0).sum())

Sesgo original de monto_compra: 95.23563302035784
Valores cero: 36712
Valores negativos: 0


In [14]:
df["monto_compra_log"] = np.log1p(df["monto_compra"])

sesgo_log = df["monto_compra_log"].skew()

print("Sesgo original:", sesgo_original)
print("Sesgo después de log1p:", sesgo_log)

Sesgo original: 95.23563302035784
Sesgo después de log1p: -2.0270014022953537


## 8. Separación de entrenamiento y prueba

Ahora que todas las transformaciones que deben existir en el dataset completo ya fueron creadas, se realiza la división 80/20.

Se utiliza `random_state=42` para mantener el mismo criterio de división de las semanas anteriores.

In [15]:
from sklearn.model_selection import train_test_split

df_entrenamiento, df_prueba = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

print("Filas de entrenamiento:", len(df_entrenamiento))
print("Filas de prueba:", len(df_prueba))
print("Total:", len(df_entrenamiento) + len(df_prueba))

Filas de entrenamiento: 414400
Filas de prueba: 103601
Total: 518001


## 9. Transformación Box-Cox

Box-Cox aprende un parámetro a partir de los datos durante `fit`, por lo que el ajuste se realiza **únicamente con entrenamiento**.

Después se utiliza el mismo transformador para transformar entrenamiento y prueba.

Box-Cox requiere valores **estrictamente positivos**. Si `monto_compra` contiene cero o valores negativos, no se fuerza la transformación y se documenta el motivo.

In [16]:
from sklearn.preprocessing import PowerTransformer

tiene_valores_no_positivos = (df["monto_compra"] <= 0).any()

if not tiene_valores_no_positivos:
    transformador_boxcox = PowerTransformer(method="box-cox")

    transformador_boxcox.fit(
        df_entrenamiento[["monto_compra"]]
    )

    df_entrenamiento["monto_compra_boxcox"] = (
        transformador_boxcox.transform(
            df_entrenamiento[["monto_compra"]]
        )
    )

    df_prueba["monto_compra_boxcox"] = (
        transformador_boxcox.transform(
            df_prueba[["monto_compra"]]
        )
    )

    sesgo_boxcox = df_entrenamiento["monto_compra_boxcox"].skew()

    print("Box-Cox se ajustó únicamente con entrenamiento.")
    print("Sesgo Box-Cox en entrenamiento:", sesgo_boxcox)

else:
    df_entrenamiento["monto_compra_boxcox"] = np.nan
    df_prueba["monto_compra_boxcox"] = np.nan
    sesgo_boxcox = np.nan

    print("Box-Cox no se aplicó.")
    print("Motivo: monto_compra contiene valores cero o negativos.")

Box-Cox no se aplicó.
Motivo: monto_compra contiene valores cero o negativos.


### ¿Por qué Box-Cox se ajusta solo con entrenamiento?

Box-Cox aprende un parámetro a partir de la distribución de los datos durante el `fit`. Por eso, el conjunto de prueba no debe participar en ese aprendizaje.

El procedimiento correcto es:

**Entrenamiento → `fit` → `transform` en entrenamiento y prueba**

De esta forma se evita la fuga de información desde el conjunto de prueba.

## 10. Comparación del sesgo

Se comparan los valores de sesgo de `monto_compra` original, después de `log1p()` y después de Box-Cox cuando esta transformación fue válida.

In [17]:
comparacion_sesgo = pd.DataFrame({
    "Transformación": [
        "Original",
        "Logarítmica (log1p)",
        "Box-Cox"
    ],
    "Sesgo": [
        sesgo_original,
        sesgo_log,
        sesgo_boxcox
    ]
})

comparacion_sesgo

,Transformación,Sesgo
0,Original,95.235633
1,Logarítmica (log1p),-2.027001
2,Box-Cox,NaN


### Interpretación de la comparación

Un valor de sesgo más cercano a 0 representa una distribución más simétrica.

La tabla anterior permite identificar cuál transformación redujo más la asimetría de `monto_compra`.

Si Box-Cox aparece como `NaN`, significa que no se aplicó porque la variable contenía valores que no eran estrictamente positivos.

## 11. Ensamblaje del dataset final

Se unen nuevamente los conjuntos de entrenamiento y prueba.

En este punto `monto_compra_log` ya existe en ambos conjuntos porque fue creado en `df` **antes** de la separación. `monto_compra_boxcox` también existe en ambos conjuntos porque se creó después de la separación.

In [18]:
df_final = pd.concat(
    [df_entrenamiento, df_prueba],
    axis=0
).reset_index(drop=True)

print("Dimensiones del dataset final:", df_final.shape)

Dimensiones del dataset final: (518001, 64)


## 12. Verificación de las nuevas variables

Se comprueba que todas las columnas solicitadas estén presentes en `df_final`.

In [19]:
columnas_nuevas = [
    "mes_pedido",
    "trimestre_pedido",
    "dia_semana_pedido",
    "es_fin_semana_pedido",
    "longitud_comentario",
    "num_palabras_comentario",
    "tiene_comentario",
    "menciona_demora",
    "grupo_edad",
    "monto_compra_log",
    "monto_compra_boxcox"
]

faltantes = [
    columna for columna in columnas_nuevas
    if columna not in df_final.columns
]

print("Columnas faltantes:", faltantes)

if len(faltantes) == 0:
    print("Todas las columnas nuevas están presentes en df_final.")

Columnas faltantes: []
Todas las columnas nuevas están presentes en df_final.


In [20]:
df_final[[
    "fecha_pedido",
    "mes_pedido",
    "trimestre_pedido",
    "dia_semana_pedido",
    "es_fin_semana_pedido",
    "comentario_cliente",
    "longitud_comentario",
    "num_palabras_comentario",
    "tiene_comentario",
    "menciona_demora",
    "edad_cliente",
    "grupo_edad",
    "monto_compra",
    "monto_compra_log",
    "monto_compra_boxcox"
]].head(10)

,fecha_pedido,mes_pedido,trimestre_pedido,dia_semana_pedido,es_fin_semana_pedido,comentario_cliente,longitud_comentario,num_palabras_comentario,tiene_comentario,menciona_demora,edad_cliente,grupo_edad,monto_compra,monto_compra_log,monto_compra_boxcox
0,2025-04-22,4.0,2.0,Tuesday,False,"Todo perfecto, llegó a tiempo",29,5,True,False,26,26 a 35,1398000.0,14.150554,NaN
1,2025-08-21,8.0,3.0,Thursday,False,Rápido y sin problemas,22,4,True,False,41,36 a 45,1809300.0,14.408451,NaN
2,2025-09-11,9.0,3.0,Thursday,False,,0,0,False,False,39,36 a 45,459000.0,13.036808,NaN
3,2026-04-11,4.0,2.0,Saturday,True,El domiciliario fue muy amable,30,5,True,False,51,46 a 60,146800.0,11.896833,NaN
4,NaT,NaN,NaN,NaN,False,Rápido y sin problemas,22,4,True,False,19,18 a 25,562600.0,13.240326,NaN
5,2025-12-04,12.0,4.0,Thursday,False,Pésima experiencia con el empaque,33,5,True,False,69,61 y más,330000.0,12.706851,NaN
6,2026-04-02,4.0,2.0,Thursday,False,El domiciliario fue muy amable,30,5,True,False,64,61 y más,1809300.0,14.408451,NaN
7,NaT,NaN,NaN,NaN,False,Pésima experiencia con el empaque,33,5,True,False,41,36 a 45,467500.0,13.055157,NaN
8,NaT,NaN,NaN,NaN,False,Rápido y sin problemas,22,4,True,False,45,36 a 45,784300.0,13.572548,NaN
9,2025-06-15,6.0,2.0,Sunday,True,Muy buen precio,15,3,True,False,35,26 a 35,0.0,0.000000,NaN


## 13. Verificaciones finales

Se comprueba que las columnas originales importantes se conservaron y que el número total de filas coincide con el dataset inicial.

In [21]:
columnas_originales = [
    "fecha_pedido",
    "comentario_cliente",
    "edad_cliente",
    "monto_compra"
]

print("Columnas originales:")
for columna in columnas_originales:
    print(f"{columna}: {columna in df_final.columns}")

print("\nFilas iniciales:", len(df))
print("Filas finales:", len(df_final))
print("Misma cantidad de filas:", len(df) == len(df_final))

Columnas originales:
fecha_pedido: True
comentario_cliente: True
edad_cliente: True
monto_compra: True

Filas iniciales: 518001
Filas finales: 518001
Misma cantidad de filas: True


## 14. Exportación del dataset transformado

In [22]:
archivo_salida = "S10_PD_Sandoval_DatasetTransformado.csv"

df_final.to_csv(
    archivo_salida,
    index=False
)

print("Archivo exportado correctamente:", archivo_salida)

Archivo exportado correctamente: S10_PD_Sandoval_DatasetTransformado.csv


In [23]:
df_comprobacion = pd.read_csv(archivo_salida)

print("Dimensiones del archivo exportado:", df_comprobacion.shape)
print("Columnas nuevas disponibles:")
print([
    columna for columna in columnas_nuevas
    if columna in df_comprobacion.columns
])

Dimensiones del archivo exportado: (518001, 64)
Columnas nuevas disponibles:
['mes_pedido', 'trimestre_pedido', 'dia_semana_pedido', 'es_fin_semana_pedido', 'longitud_comentario', 'num_palabras_comentario', 'tiene_comentario', 'menciona_demora', 'grupo_edad', 'monto_compra_log', 'monto_compra_boxcox']


## Conclusión

En esta semana se transformaron variables temporales, texto libre y edad para obtener representaciones más útiles para el análisis. También se evaluó y redujo el sesgo de `monto_compra` mediante una transformación logarítmica y se evaluó la aplicación de Box-Cox respetando el principio de separar el aprendizaje de la transformación del conjunto de prueba.

El dataset final conserva las variables originales y agrega las nuevas variables derivadas necesarias para continuar con el procesamiento de datos.